<a href="https://colab.research.google.com/github/Shea-Fyffe/transforming-personality-item-generation/blob/main/template-aig-prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Import Libraries
# standard
from dataclasses import dataclass, field, fields
from typing import Any, List, Optional, Dict
from pydantic import BaseModel, create_model, Field, TypeAdapter
from langchain.prompts import PromptTemplate
from langchain.output_parsers import PydanticOutputParser

# application specific
import json  # for item params
import pandas as pd  # for prompt generation
import numpy as np  # for prompt generation

In [ ]:
# @title Utility Functions
# predicate function for testing if or iterable is empty
def is_empty(x):
    if isinstance(x, list):
        return all(is_empty(i) for i in x)
    return not bool(x)


# helper to pop and find
def extract_from_dict(kwargs: dict, pattern: str):
    extract = {}
    keep = {}
    for k, v in kwargs.items():
        if k.startswith(pattern):
            extract[k] = v
        else:
            keep[k] = v
    return extract, keep


# convert string to length one list
def str_to_list(x: str | list[str]) -> list:
    if isinstance(x, list):
        return x
    return [x]


# for mfc options
def lstr_to_list(x: list[str], delim_char: str = "|"):
    def _str_to_list(string, delim):
        if isinstance(string, str):
            if string:
                return list(string.split(delim))
        return None

    return [_str_to_list(s, delim_char) for s in x]


# get all combinations results
def combn(a: list | int, n: int = 3, sort=True) -> list[Any]:
    if isinstance(a, int):
        a = list(range(a))
    if sort:
        a = sorted(a)
    return [list(i) for i in _combn(a, n)]


def _combn(iterable, r):
    pool = tuple(iterable)
    n = len(pool)
    if r > n:
        return
    indices = list(range(r))
    yield tuple(pool[i] for i in indices)
    while True:
        for i in reversed(range(r)):
            if indices[i] != i + n - r:
                break
        else:
            return
        indices[i] += 1
        for j in range(i + 1, r):
            indices[j] = indices[j - 1] + 1
        yield tuple(pool[i] for i in indices)


# prompt id generator
def make_prompt_id(item_types: list[str], prepend_char: None | str = "PR") -> list[str]:
    return [
        prepend_char + v[:3].upper() + "%0.3d" % (i + 1)
        for i, v in enumerate(item_types)
    ]

In [ ]:
# @title Template Class
@dataclass
class PromptObj:
    item_type: str
    item_construct: str | list[str] | None
    item_context: str = "general"
    item_count: int = 1
    example_items: str | list[str] | None = None
    item_kwargs: dict[str, Any] | None = field(default_factory=dict)
    option_kwargs: dict[str, Any] | None = field(default_factory=dict)
    base_constraints: str | list[str] | None = field(
        default_factory=lambda: [
            "Omit `$defs` and any additional JSON schema information from the output."
        ]
    )
    additional_constraints: str | list[str] | None = None
    generation_template: str = (
        """{instructions}\n\n# Item Specifications as JSON:\n{item_specifications}\n\n# Example Items:\n{example_items}\n\n# Output Format:\n{format_instructions}\n\n# Constraints:\n{constraints}\n"""
    )
    kwargs: dict[str, Any] | None = field(default_factory=dict)

    ## Post Init ##
    def __post_init__(self) -> None:
        # validate keyword arguments
        self.sanitize_kwargs()

    ## Class Methods ##
    @classmethod
    def as_likert(
        cls,
        item_construct: str,
        item_context: str = "general",
        item_count: int = 1,
        example_items=None,
        **kwargs,
    ):

        return cls(
            item_type="likert",
            item_construct=item_construct,
            item_context=item_context,
            item_count=item_count,
            example_items=example_items,
            kwargs={**kwargs},
        )

    @classmethod
    def as_mfc(
        cls,
        item_context: str = "general",
        item_count: int = 1,
        example_items=None,
        item_stem=None,
        option_constructs: list[str] = [
            "conscientiousness",
            "extraversion",
            "agreeableness",
        ],
        option_count=3,
        additional_constraints: str | list[str] = [
            "Options choices should have similar levels of social desirability"
        ],
        **kwargs,
    ):

        return cls(
            item_type="multidimensional forced-choice",
            item_construct=option_constructs,
            item_context=item_context,
            item_count=item_count,
            example_items=example_items,
            additional_constraints=additional_constraints,
            option_kwargs={
                "option_count": option_count,
                "option_constructs": option_constructs,
            },
            kwargs={**kwargs},
        )

    @classmethod
    def as_sjt(
        cls,
        item_construct: str,
        item_context: str = "general",
        item_count: int = 1,
        example_items=None,
        option_count=4,
        additional_constraints: str | list[str] = [
            "Options choices should represent different degrees of the item construct."
        ],
        **kwargs,
    ):

        return cls(
            item_type="situational judgement test",
            item_construct=item_construct,
            item_context=item_context,
            item_count=item_count,
            example_items=example_items,
            additional_constraints=additional_constraints,
            option_kwargs={"option_count": option_count},
            kwargs={**kwargs},
        )

    ## Instance Methods ##
    def list_to_bulleted_str(self, x: list[str], quote: bool = False) -> str | None:
        if (not x) or (not isinstance(x, list)):
            return None
        return "\n".join(
            [f'{i + 1}. "{v}"' if quote else f"- {v}" for i, v in enumerate(x)]
        )

    ### Arguments
    def processes_etc_kwargs(self) -> None:
        """
        Instantiate addition kwargs for further customization.
        """
        self.max_item_examples = 5
        if "max_item_examples" in self.kwargs.keys():
            self.max_item_examples = self.kwargs.pop("max_item_examples")
        # ... list additional kwargs here

    def update_item_kwargs(self) -> None:
        """
        Passes any kwargs starting with 'item' to `item_kwargs`
        """
        new_kwargs, _ = extract_from_dict(self.kwargs, "item")
        if new_kwargs:
            self.item_kwargs.update(new_kwargs)

    def update_option_kwargs(self) -> None:
        """
        Passes any kwargs starting with 'option' to `option_kwargs`
        """
        new_kwargs, _ = extract_from_dict(self.kwargs, "option")
        if new_kwargs:
            self.option_kwargs.update(new_kwargs)

    def sanitize_kwargs(self) -> None:
        """
        Validates keyword arugments.
        """
        self.processes_etc_kwargs()
        if self.item_kwargs is None:
            self.item_kwargs = {}
        if self.option_kwargs is None:
            self.option_kwargs = {}
        if not is_empty(self.kwargs):
            self.update_item_kwargs()
            self.update_option_kwargs()
        for key in self.get_required_params.keys():
            if self.item_kwargs and key in self.item_kwargs.keys():
                self.item_kwargs.pop(key)

    ### Formatters
    def format_item_specifications(self) -> None:
        """
        Validates and processes item specification arguments.
        """
        self._item_specifications = {}
        if self.has_required_params:
            self._item_specifications.update(self.get_required_params)
        if self.has_kwargs:
            self._item_specifications.update(
                {**self.item_kwargs, "options": {**self.option_kwargs}, **self.kwargs}
            )

    def format_example_items(self) -> None:
        """
        Validates and processes any items provided as examples.
        """
        if self.has_example_items:
            if isinstance(self.example_items, dict):
                self.example_items = self.example_items.values()
            self.example_items = [i.strip() for i in str_to_list(self.example_items)]
            if len(self.example_items) > self.max_item_examples:
                self._example_items = [
                    self.example_items[i] for i in range(self.max_item_examples)
                ]
            else:
                self._example_items = self.example_items
        else:
            self._example_items = self.get_pseudo_item_examples
        self._example_items = list(set(self._example_items))

    def format_constraints(self) -> None:
        """
        Combines base and additional constraints passed to the model.
        """
        self._constraints = []
        if self.base_constraints:
            self._constraints.extend(str_to_list(self.base_constraints))
        if self.additional_constraints:
            self._constraints.extend(str_to_list(self.additional_constraints))
        if self.has_example_items:
            self._constraints.extend(
                ["Do **not** generate items appearing in Example Items."]
            )
        self._constraints = list(set(self._constraints))

    def format_response_schema(self) -> None:
        self._response_schema = create_model(
            "ItemSchema", **self.ItemSchemaObj[self.item_type]
        )

    ### Setup
    def setup_prompt_instructions(self) -> str:
        return f"Generate ({self.item_count}) {self.item_type.capitalize()} items that measure {self.item_desc_inst}."

    def setup_item_specifications(self) -> str:
        self.format_item_specifications()
        return self.to_json_str(self._item_specifications)

    def setup_example_items(self) -> str | None:
        self.format_example_items()
        return self.list_to_bulleted_str(self._example_items, quote=True)

    def setup_item_constraints(self) -> str | None:
        self.format_constraints()
        return self.list_to_bulleted_str(self._constraints)

    def setup_response_schema(self) -> str:
        """
        Build the output schema for the prompt.
        """
        self.format_response_schema()
        self._schema_parser = PydanticOutputParser(
            pydantic_object=self.get_response_schema
        )
        return self._schema_parser.get_format_instructions()

    def setup_prompt(self, **kwargs) -> None:
        """
        Build the output schema for the prompt.
        """
        _defaults = {
            "template": self.generation_template,
            "input_variables": [
                "instructions",
                "item_specifications",
                "example_items",
                "constraints",
            ],
            "partial_variables": {"format_instructions": self.setup_response_schema()},
        }
        for k, v in kwargs.items():
            if k in _defaults.keys():
                _defaults[k] = v
        self._prompt_factory = PromptTemplate(**_defaults)

    ### Generators
    def make_prompt(self, prompt_setup_kwargs: dict = {}):
        self.setup_prompt(**prompt_setup_kwargs)
        _callargs = {
            "instructions": self.setup_prompt_instructions(),
            "item_specifications": self.setup_item_specifications(),
            "example_items": self.setup_example_items(),
            "constraints": self.setup_item_constraints(),
        }
        return self._prompt_factory.format_prompt(**_callargs)

    def to_json_str(self, obj: dict) -> str | None:
        try:
            return json.dumps({k: v for k, v in obj.items() if not is_empty(v)})
        except Exception as invalid_dict_for_json:
            if hasattr(invalid_dict_for_json, "add_note"):
                invalid_dict_for_json.add_note(self.json_error)
            raise

    ## Properties ##
    ### Schema Objects
    @property
    def LikertJSONSchema(self) -> dict:
        return {
            "item_text": (
                str,
                Field(
                    description="A Likert statement representing a personality or psychological characteristic to which a person can agree or disagree"
                ),
            ),
            "item_construct": (
                Optional[str],
                Field(
                    None,
                    description="The personality trait or psychological characteristic an item measures",
                ),
            ),
            "item_context": (
                Optional[str],
                Field(
                    None,
                    description="The situation or environment the item relates to",
                    examples=["general", "work", "school", "military"],
                ),
            ),
        }

    @property
    def MFCJSONSchema(self) -> dict:
        return {
            "item_text": (
                str,
                Field(
                    "Rank the following options from Most Like You (1) to Least Like You (3):",
                    description="The text of a multidimensional-forced choice item",
                ),
            ),
            "option_choices": (
                List[str],
                Field(
                    description="Statements of similar social desirability that measure each of the option constructs",
                    min_items=self.get_option_count,
                    max_items=self.get_option_count,
                ),
            ),
            "option_constructs": (
                List[str],
                Field(
                    description="The personality trait or psychological characteristic an option choice measures",
                    min_items=self.get_option_count,
                    max_items=self.get_option_count,
                ),
            ),
            "item_context": (
                Optional[str],
                Field(
                    None,
                    description="The situation or environment the item relates to",
                    examples=["general", "work", "school", "military"],
                ),
            ),
        }

    @property
    def SJTJSONSchema(self) -> dict:
        return {
            "item_text": (
                str,
                Field(description="The text of a situational judgement test item"),
            ),
            "option_choices": (
                List[str],
                Field(
                    description="Behavioral responses or reactions to the situation described by the item text which vary in terms of degrees of the item construct",
                    min_items=self.get_option_count,
                    max_items=self.get_option_count,
                ),
            ),
            "item_construct": (
                str,
                Field(
                    description="The personality trait or psychological characteristic an item measures"
                ),
            ),
            "item_context": (
                Optional[str],
                Field(
                    None,
                    description="The situation or environment the item relates to",
                    examples=["general", "work", "school", "military"],
                ),
            ),
        }

    @property
    def ItemSchemaObj(self) -> dict:
        return {
            "likert": self.LikertJSONSchema,
            "multidimensional forced-choice": self.MFCJSONSchema,
            "situational judgement test": self.SJTJSONSchema,
        }

    @property
    def get_response_schema(self) -> BaseModel:
        return create_model(
            "Assessment Scale",
            ScaleItemList=(
                List[self._response_schema],
                Field(
                    description="A list or array of scale items", alias="ScaleItemList"
                ),
            ),
        )

    ### Validators
    @property
    def json_error(self) -> str:
        err_str = [
            "item_type, item_constuct, item_context, item_kwargs, or option_kwargs",
            "values are invalid in a json format.",
            "Input this objects item_params_dict attritbute into json.dumps()",
            "for a more informative error.",
        ]
        return " ".join(err_str)

    @property
    def has_example_items(self) -> bool:
        return not is_empty(self.example_items)

    @property
    def has_required_params(self) -> bool:
        return all([not is_empty(v) for v in self.get_required_params.values()])

    @property
    def has_kwargs(self) -> bool:
        return not (
            is_empty(self.kwargs)
            & is_empty(self.item_kwargs)
            & is_empty(self.option_kwargs)
        )

    ### Getters
    @property
    def get_required_params(self) -> dict:
        return {
            "item_type": self.item_type,
            "item_construct": self.item_construct,
            "item_context": self.item_context,
            "item_count": self.item_count,
        }

    @property
    def get_base_constraints(self) -> list[str | None]:
        return self.base_constraints

    @property
    def get_constraints(self) -> list[str | None]:
        if hasattr(self, "_constraints"):
            return self._constraints
        return self.get_base_constraints

    @property
    def get_constructs(self) -> list[str] | str:
        if self.item_type == "multidimensional forced-choice":
            return self.option_kwargs["option_constructs"]
        return self.item_construct

    @property
    def get_item_stem(self) -> str | None:
        self.item_kwargs["item_stem"] if "item_stem" in self.item_kwargs.keys() else ""

    @property
    def get_item_count(self) -> int:
        if "item_count" in self.item_kwargs.keys():
            return self.item_kwargs["item_count"]
        else:
            return 1

    @property
    def get_option_count(self) -> int:
        if "option_count" in self.option_kwargs.keys():
            return self.option_kwargs["option_count"]
        else:
            return 1

    ### Helpers
    @property
    def item_desc_inst(self) -> str | None:
        return f"{self.construct_str} in a {self.item_context} context"

    @property
    def item_desc_ex(self) -> str | None:
        _const = self.construct_str.replace(" and ", " or ")
        _type = self.item_type + " item"
        if self.get_option_count > 1:
            _type = f"{_type} with {self.get_option_count} option choices"
        if "item_keys" in self.item_kwargs.keys():
            _type = f'{self.item_kwargs["item_keys"]}ly keyed {_type}'
        return f"{_type} measuring {_const}"

    @property
    def construct_str(self) -> str:
        if self.item_type == "multidimensional forced-choice":
            _opts = [i.capitalize() for i in self.get_constructs]
            return f'{", ".join(_opts[:-1])} and {_opts[-1]}'
        return self.get_constructs.capitalize()

    @property
    def pseudo_examples(self) -> list[str]:
        if self.get_item_count > 1:
            self.max_item_examples = (
                10 if not hasattr(self, "max_item_examples") else self.max_item_examples
            )
            _k = min(self.max_item_examples, self.get_item_count)
            return [
                (
                    "`%%another " + self.item_desc_ex + "%%`"
                    if i > 0
                    else "`%%a " + self.item_desc_ex + "%%`"
                )
                for i in range(_k)
            ]
        return ["`%%a " + self.item_desc_ex + "%%`"]

    @property
    def get_pseudo_item_examples(self) -> list[str]:
        return self.pseudo_examples

In [ ]:
# @title Prompt Templating Objects
def GET_BIG5_DATA():
    ITEM_TYPE = [
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "sjt",
        "sjt",
        "sjt",
        "sjt",
        "sjt",
        "sjt",
        "sjt",
        "mfc",
        "mfc",
        "mfc",
        "mfc",
        "mfc",
        "mfc",
        "mfc",
        "mfc",
        "mfc",
        "mfc",
    ]
    ITEM_CONTEXT = [
        "general",
        "general",
        "general",
        "general",
        "general",
        "general",
        "general",
        "general",
        "general",
        "general",
        "work",
        "work",
        "work",
        "school",
        "school",
        "general",
        "general",
        "general",
        "general",
        "general",
        "general",
        "general",
        "general",
        "general",
        "general",
        "general",
        "general",
    ]
    ITEM_CONSTRUCT = [
        "agreeableness",
        "agreeableness",
        "conscientiousness",
        "conscientiousness",
        "extraversion",
        "extraversion",
        "neuroticism",
        "neuroticism",
        "openness",
        "openness",
        "agreeableness",
        "conscientiousness",
        "extraversion",
        "neuroticism",
        "openness",
        "neuroticism",
        "agreeableness",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
    ]
    ITEM_COUNT = [
        3,
        9,
        3,
        9,
        3,
        9,
        3,
        9,
        3,
        9,
        2,
        2,
        2,
        2,
        3,
        1,
        3,
        2,
        2,
        2,
        2,
        2,
        2,
        2,
        2,
        2,
        2,
    ]
    ITEM_WORDING = [
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
    ]
    OPTION_CONSTRUCT = [
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "agreeableness|conscientiousness|extraversion",
        "agreeableness|conscientiousness|neuroticism",
        "agreeableness|conscientiousness|openness",
        "agreeableness|extraversion|neuroticism",
        "agreeableness|extraversion|openness",
        "agreeableness|neuroticism|openness",
        "conscientiousness|extraversion|neuroticism",
        "conscientiousness|extraversion|openness",
        "conscientiousness|neuroticism|openness",
        "extraversion|neuroticism|openness",
    ]
    OPTION_COUNT = [
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        4,
        4,
        4,
        4,
        5,
        5,
        4,
        3,
        3,
        3,
        3,
        3,
        3,
        3,
        3,
        3,
        3,
    ]
    EXAMPLE_ITEMS = [
        ["I am cold.", "I am unkind."],
        [
            "I am kind",
            "I am cooperative",
            "I am sympathetic",
            "I am warm",
            "I am trustful",
            "I am considerate.",
        ],
        ["I am disorganized", "I am careless."],
        [
            "I am organized",
            "I am systematic",
            "I am thorough",
            "I am practical",
            "I am neat",
            "I am efficient.",
        ],
        ["I am introverted", "I am shy."],
        [
            "I am extraverted",
            "I am talkative",
            "I am assertive",
            "I am verbal",
            "I am energetic",
            "I am bold.",
        ],
        ["I am not easily annoyed", "I feel comfortable with myself."],
        [
            "I worry about things",
            "I fear for the worst",
            "I am afraid of many things",
            "I get stressed out easily",
            "I get angry easily",
            "I get irritated easily.",
        ],
        ["I do not like poetry", "I do not enjoy going to art museums."],
        [
            "I have a vivid imagination",
            "I enjoy wild flights of fantasy",
            "I love to daydream",
            "I like to get lost in thought",
            "I believe in the importance of art",
            "I see beauty in things that others might not notice.",
        ],
        [
            "Last year one of your colleagues took credit for business you brought in. You are about to start a new collaborative project with this colleague. What would you do?\n- I would have long forgotten about the incident and would start this new collaborative project with an open attitude.\n- I would still remember the incident, ut a year is a long time. I would give him a second chance.\n- I would remember the incident as if it were yesterday. I would tell him so, ut still give him a second chance.\n- I would not be able to forget about the incident and therefore decide not to pursue this collaboration",
            "A coworker of the department that you are supervising made an important phone call with a client. You had told the coworker in advance that this was not his task and that you would make the phone call yourself. The phone call with the client was unsuccessful and now you have missed out on an important deal. What would you do?\n- I would be terribly upset about the fact that he made the phone call behind my back and about losing the client. I would firmly address him and tell him that his actions have consequences.\n- I would ask the coworker to come to my office and calmly tell him that I do not appreciate his actions.\n- I would ask the coworker to come to my office. I would tell him that I forgive him and ask what went wrong during the phone call.\n- I would explain that I understand that coworkers find it important to have these kinds of phone calls with their own clients. I decide to give a negotiations training to all coworkers.",
        ],
        [
            "You have finished a proposal that has to be submitted by tomorrow. The secretary usually checks these proposals for errors. What would you do?\n- I would send the proposal to the secretary and I would submit it after her check.\n- As soon as the secretary sends back the proposal I would give it another quick check and then submit it.\n- I would check the proposal once more before I send it to the secretary. When the secretary sends back the proposal I would give it another check and then submit it.\n- I would give it a thorough check before I send it to the secretary. When the secretary sends back the proposal I would give it another thorough check and then submit it",
            "You have to give a presentation to your team soon, or which you are making PowerPoint slides. What would you do?\n- I would only spend time on the content; details regarding layout and language are not important.\n- I would spend a lot of time on the content and less on the details regarding layout and language.\n- I would spend a lot of time on the content, ut also some time on the details regarding layout and language.\n- I would spend a lot of time on the content, ut at least as much time on the details regarding layout and language.",
        ],
        [
            "During a meeting, he chair asks people to share some workplace experiences. What would you do?\n- I would be the first to start talking and elaborate on my experiences.\n- After one of my colleagues has said something, would say something too.\n- I would only say something if I am asked personally to share my experiences.\n- I would be hoping for my colleagues to say something so that I do not have to",
            "You just had two weeks of holiday and it is time to get back to work. How would you feel?\n- I would dread having to answer questions about my holiday again and again and therefore I would try to avoid my colleagues on the first day.\n- Although I would look forward to seeing my colleagues again, notice that all the attention costs me energy.\n- I would enjoy seeing everyone again and to share my holiday experiences with my colleagues.\n- I would be full of energy and stories and share them with all my colleagues.",
        ],
        [
            "Tomorrow you will have your performance review, and there is a lot at stake. It is late in the evening and you are feeling tired. What would you do?\n- I would go to bed and fall asleep right away even though I am a bit anxious.\n- I would go to bed but would have trouble falling asleep, nce I fall asleep I manage to have a good night's rest.\n- I would go to bed but lie awake for hours and sleep poorly.\n- I would go to bed but lie awake all night worrying about the next day",
            "You have applied for a desirable job and you will hear soon whether they will make you an offer. What would you do?\n- I would calmly wait until I hear something.\n- I would be a bit nervous, ut focus on other things.\n- I would be nervous and have difficulty concentrating on other things.\n- I would be terribly nervous and unable to think of anything else.",
        ],
        [
            "You have accepted a scholarship to play a sport at your dream school. During your first semester, t turns out that the school and team are not what you expected; in fact, ou are quite disappointed. You are having trouble adjusting to the college experience and are struggling in your classes. What would you most likely do? What would you least likely do?\n- Tell yourself that you cannot balance being a student and an athlete at the same time\n- Tell yourself that a bad semester is ok and that next semester things will work out\n- Decide to withdraw either from the school, he team, r both\n- Talk to your parents and peers about your feelings about the current situation\n- Simply accept that this is what college is going to be like and carry on",
            "As a law student in a prestigious law school, ou are under intense pressure and doing your utmost to get ahead of your peers. At the end of your first year in law school, ou got your annual ranking and you learned that you were ranked at the bottom 30th percentile. What would you most likely do? What would you least likely do?\n- Contact your peers and form a study group\n- Call your friends to decompress\n- Give up an additional 2 hours of sleep so you can spend more time studying\n- Decide that pursuing a J.D. degree is not for you and change your career path\n- Realize that it is just your first year; things will work out well in the end",
            "You are taking an upper level physics class to fulfill a college major requirement. It is one of the hardest courses you have taken and is regarded as a good indicator of future career success. Halfway through the semester, ou have completed two course projects and you got C's for both. What would you most likely do? What would you least likely do?\n- Tell yourself that you will do well in the end because you are smart and hard-working\n- Go to your professor to ask for feedback and study accordingly\n- Decide to switch to a less demanding field since you do not perform well in this one\n- Tell yourself that this is just a minor setback and things will turn out well in the end\n- Talk to other students in your class for support",
        ],
        [
            "A good friend of yours has a new significant other. Lately this friend has been canceling plans to meet you in order to spend time with their new love. After meeting for lunch one day, our friend tells you they miss spending time with you and apologizes for not being able to see you as much. You make plans, ut your friend cancels again at the last minute and explains they are going with their significant other on a romantic getaway. What would you most likely do? What would you least likely do?\n- Let your friend know that you are done making plans with him/her\n- Call a friend to explain what has been going on and ask for advice about what to do\n- Remind yourself to be happy for your love-struck friend and expect that eventually you'll be close again\n- Make an effort to plan around his/her schedule\n- Tell yourself you can handle this situation; you've handled worse"
        ],
        [
            "Andre moves away from the city his friends and family are in. He finds his friends make less effort to keep in contact than he thought they would. What action would be the most effective for Andre?\n- Try to adjust to life in the new city by joining clubs and activities there.\n- He should make the effort to contact them, ut also try to meet people in his new city.\n- Let go of his old friends, ho have shown themselves to be unreliable.\n- Tell his friends he is disappointed in them for not contacting him",
            "Clayton has been overseas for a long time and returns to visit his family. So much has changed that Clayton feels left out. What action would be the most effective for Clayton?\n- Nothing--it will sort itself out soon enough.\n- Tell his family he feels left out.\n- Spend time listening and getting involved again.\n- Reflect that relationships can change with time",
            "Katerina takes a long time to set the DVD timer. With the family watching, er sister says \"You idiot, ou're doing it all wrong, an't you work the video?\" Katerina is quite close to her sister and family. What action would be the most effective for Katerina?\n- Ignore her sister and keep at the task.\n- Get her sister to help or to do it.\n- Tell her sister she is being mean.\n- Never work appliances in front of her sister or family again.",
        ],
        [
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I sympathize with others' feelings.\n- I pay my bills on time.\n- I make friends easily",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I accept people as they are.\n- I am always prepared.\n- I am the life of the party",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I trust what people say.\n- I do things according to a plan.\n- I feel comfortable around people",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I get back at others.\n- I waste my time.\n- I find it difficult to approach others.",
        ],
        [
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I see the good in people.\n- I organize my time.\n- I am relaxed most of the time",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I pay attention to the needs of others.\n- I always perform tasks conscientiously.\n- I seldom get mad.",
        ],
        [
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I respect others.\n- I follow through with my plans.\n- I have a rich vocabulary",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I have a good word for everyone.\n- I am exacting in my work.\n- I have a vivid imagination",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I sympathize with others' feelings.\n- I get chores done right away.\n- I enjoy wild flights of fantasy.",
        ],
        [
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I distrust people.\n- I have little to say.\n- I get stressed out easily",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I am not interested in other people's problems.\n- I am a loner.\n- I dislike myself.",
        ],
        [
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I don't allow others to use me.\n- I find it easy to get my way.\n- I am not interested in abstract ideas",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I treat all people equally.\n- I warm up quickly to others.\n- I get excited by new ideas",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I insult people.\n- I don't like to draw attention to myself.\n- I am interested in theoretical discussions.",
        ],
        [
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I trust others.\n- I recover quickly from stress and illness.\n- I am very sensitive",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I am concerned about others.\n- I feel comfortable with myself.\n- I believe in the importance of art.",
        ],
        [
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I act without thinking.\n- I find it hard to approach others.\n- I am often sad",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I carry out everything according to plan.\n- I think it is exciting to talk to many different people.\n- I feel comfortable with myself",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I don't feel comfortable when everything is a mess.\n- I talk to a lot of different people at parties.\n- I am very pleased with myself",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I follow a schedule.\n- I know how to captivate people.\n- I am an optimist",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I complete tasks successfully.\n- I talk to a lot of different people at parties.\n- I am relaxed most of the time.",
        ],
        [
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I tend to be very particular about things.\n- I stay in the background.\n- I have a vivid imagination",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I get chores done right away.\n- I am the life of the party.\n- I don't let my emotions overwhelm me easily",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I carry out my plans.\n- I cheer people up.\n- I rarely look for a deeper meaning in things.",
        ],
        [
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I like to tidy up.\n- I am always in a good mood.\n- I love to think up new ways of doing things",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I like it when everything is in its place.\n- I stay calm in difficult situations.\n- I am full of ideas",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I am always prepared.\n- I am emotionally stable.\n- I like to explore new things",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I like to plan ahead.\n- I am relaxed.\n- I have a creative mind",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I find it difficult to get down to work.\n- I am often down in the dumps.\n- I do not enjoy going to art museums",
        ],
        [
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I like to talk to strangers.\n- I worry about things.\n- I have difficulty imagining things",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I love big parties.\n- I rarely worry.\n- I am imaginative",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I know how to captivate people.\n- I do not worry about things.\n- I enjoy thinking about things",
            "Rank the following options from Most Like You (1) to Least Like You (3):\n- I keep in the background.\n- I fear for the worst.\n- I am not interested in abstract ideas.",
        ],
    ]

    return {
        "prompt_id": make_prompt_id(ITEM_TYPE),
        "item_type": ITEM_TYPE,
        "item_construct": ITEM_CONSTRUCT,
        "item_context": ITEM_CONTEXT,
        "item_count": ITEM_COUNT,
        "item_wording": ITEM_WORDING,
        "example_items": EXAMPLE_ITEMS,
        "option_count": OPTION_COUNT,
        "option_constructs": lstr_to_list(OPTION_CONSTRUCT),
    }


def GET_ECT_DATA():
    ITEM_TYPE = [
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "likert",
        "mfc",
        "mfc",
        "mfc",
        "mfc",
        "sjt",
        "sjt",
        "sjt",
        "sjt",
        "sjt",
        "sjt",
        "sjt",
        "sjt",
        "sjt",
        "sjt",
    ]
    ITEM_CONSTRUCT = [
        "adaptability",
        "emotion regulation",
        "optimism",
        "self-efficacy",
        "boldness",
        "compulsivity",
        "creativity",
        "impression management",
        "interpersonal dominance",
        "negative affect",
        "positive affect",
        "psychoticism",
        "resilience",
        "ruthlessness",
        "self-efficacy",
        "tolerance for uncertainty",
        "",
        "",
        "",
        "",
        "emotional intelligence",
        "resilience",
        "emotional intelligence",
        "resilience",
        "emotional intelligence",
        "honesty-humility",
        "integrity",
        "resilience",
        "social appraising",
        "social scanning",
    ]
    ITEM_CONTEXT = [
        "general",
        "general",
        "general",
        "general",
        "work",
        "work",
        "work",
        "work",
        "work",
        "work",
        "work",
        "work",
        "work",
        "work",
        "work",
        "work",
        "work",
        "work",
        "work",
        "work",
        "general",
        "general",
        "school",
        "school",
        "work",
        "work",
        "work",
        "work",
        "work",
        "work",
    ]
    OPTION_CONSTRUCT = [
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "antagonism|compulsivity|detachment",
        "compulsivity|disinhibition|psychoticism",
        "detachment|disinhibition|psychoticism",
        "detachment|negative affect|psychoticism",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
    ]
    ITEM_COUNT = [
        5,
        3,
        4,
        5,
        12,
        8,
        4,
        9,
        9,
        9,
        5,
        7,
        2,
        6,
        7,
        3,
        1,
        1,
        2,
        1,
        10,
        8,
        2,
        8,
        8,
        2,
        7,
        5,
        4,
        3,
    ]
    OPTION_COUNT = [
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        3,
        3,
        3,
        3,
        4,
        5,
        4,
        6,
        4,
        4,
        4,
        5,
        6,
        6,
    ]

    return {
        "prompt_id": make_prompt_id(ITEM_TYPE, prepend_char="OPR"),
        "item_type": ITEM_TYPE,
        "item_construct": ITEM_CONSTRUCT,
        "item_context": ITEM_CONTEXT,
        "item_count": ITEM_COUNT,
        "option_count": OPTION_COUNT,
        "option_constructs": lstr_to_list(OPTION_CONSTRUCT),
    }


def GET_AIG_DATA(
    input_params: list[dict] | dict = [GET_BIG5_DATA(), GET_ECT_DATA()],
    fill_missing_with=None,
    missing_vals=["", np.nan, pd.NA],
):
    def dfill(dlist):
        def _dfill(d, key_index):
            for ki in key_index:
                if ki not in d.keys():
                    d[ki] = [fill_missing_with]
            return d

        all_keys = list(set().union(*(d.keys() for d in dlist)))
        if isinstance(dlist, dict):
            return [dlist]
        return [_dfill(d, all_keys) for d in dlist]

    return (
        pd.concat([pd.DataFrame(i) for i in input_params])
        .reset_index(drop=True)
        .replace(missing_vals, fill_missing_with)
    )


def PROMPT_GENERATOR(
    params_data: pd.DataFrame,
    filter_kwargs: None | str | list[str] = ["item_type", "prompt_id"],
    debug: bool = False,
):
    assert isinstance(params_data, pd.DataFrame)
    if isinstance(filter_kwargs, str):
        filter_kwargs = [filter_kwargs]
    fun_dict = {
        "likert": PromptObj.as_likert,
        "mfc": PromptObj.as_mfc,
        "sjt": PromptObj.as_sjt,
    }
    res = []
    for i in params_data.itertuples(index=False):
        _init_fun = fun_dict.get(i.item_type, "")
        _args = {
            k: v
            for k, v in i._asdict().items()
            if not is_empty(v) and k not in filter_kwargs
        }
        _prompt_obj = _init_fun(**_args)
        prompt_str = _prompt_obj.make_prompt()
        res.append(
            {
                "prompt_id": i.prompt_id,
                "prompt_obj": _prompt_obj,
                "prompt_str": prompt_str.text,
            }
        )
    return res


def PROMPTS_TO_JSON(prompts: list[dict], **kwargs) -> list[dict]:
    return [
        {"id": i["prompt_id"], "prompt": i["prompt_str"], "output": [], **kwargs}
        for i in prompts
    ]

In [ ]:
# @title Generate AIG Prompt Templates
PROMPT_TEMPLATES = PROMPT_GENERATOR(GET_AIG_DATA())

In [ ]:
# @title Write to JSON File
OUTFILE_PATH = "aig-prompt-templates.json"
with open(OUTFILE_PATH, "w") as outfile:
    json.dump(PROMPTS_TO_JSON(PROMPT_TEMPLATES), outfile)